# Privacy Auditing with One (1) Training Run

**Paper:** Steinke, Nasr & Jagielski (2023) — *Privacy Auditing with One (1) Training Run*

**Implements:** Algorithm 1 + Algorithm 3 (black-box, input-space, loss-difference scoring)

**Server constraints:** No pip/internet. All packages pre-installed via wheels. Dataset loaded from local tar.gz.

### What this notebook does (one training run per shadow model)
1. Loads CIFAR-10 from local `.tar.gz` — no download needed
2. For each shadow model (Algorithm 1):
   - Flip m independent coins → each canary is IN or OUT
   - **Save `w_0` (initial weights) before any training**
   - Train WideResNet-16-4 with DP-SGD (Opacus)
   - Score: `Score(xᵢ) = loss(w_0, xᵢ) − loss(w_T, xᵢ)` (Algorithm 3, line 9)
   - Guess IN = top-k scores, OUT = bottom-k scores
3. Run statistical audit → provable ε lower bound (Corollary 5.4)
4. Aggregate across shadow models, report best bound

### Key fixes vs. previous version
- ✅ Score = `loss(w_0, x) − loss(w_T, x)` not `-final_loss`
- ✅ `random_crop` dead-code bug fixed (was returning after pad only)
- ✅ IN/OUT split logic simplified and verified
- ✅ No network calls anywhere — fully offline

---
## § 1 — Imports & Device

In [1]:
# ── No pip installs — everything must be pre-installed on this server ─────────
# Required packages (installed via wheels before launching JupyterLab):
#   torch, torchvision, opacus>=1.5, scipy, tqdm, matplotlib

import os, math, copy, random, json, pathlib, warnings, time, sys
warnings.filterwarnings("ignore")

os.environ["CUDA_VISIBLE_DEVICES"] = "1" # 🟢 ADD THIS LINE: Locks to GPU 1
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import scipy.stats
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.transforms.functional as TFF
import tqdm
import matplotlib.pyplot as plt

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float32

# CIFAR-10 normalisation constants (per-channel mean/std from training set)
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"Opacus  : {__import__('opacus').__version__}")
print(f"Device  : {DEVICE}  (CUDA GPUs: {torch.cuda.device_count()})")

Python  : 3.10.12
PyTorch : 2.5.1+cu124
Opacus  : 1.5.4
Device  : cuda  (CUDA GPUs: 1)


---
## § 2 — Configuration

All hyperparameters in one dict.  Edit here only.

In [2]:
import pathlib

CFG = {
    # ── Paths ──────────────────────────────────────────────────────────────────
    "data_tar"    : pathlib.Path("./data/cifar-10-python.tar.gz"),
    "data_dir"    : pathlib.Path("./data"),

    # ── Audit (Algorithm 1) ────────────────────────────────────────────────────
    "num_canaries" : 3000,
    "num_shadow"   : 6,
    "canary_type"  : "random_labels",
    "seed"         : 2024,

    # ── DP-SGD ────────────────────────────────────────────────────────────────
    "batch_size"      : 4096,
    "max_phys_batch"  : 1024,
    "num_epochs"      : 150,
    "learning_rate"   : 4.0,
    "noise_multiplier": 1.5,
    "max_grad_norm"   : 1.0,
    "delta"           : 1e-5,
    "num_workers"     : 4,

    # ── Audit evaluation ──────────────────────────────────────────────────────
    "k_plus"       : 450,
    "k_minus"      : 450,
    "confidence_p" : 0.05,
}

# ── Auto-generated experiment name ───────────────────────────────────────────
exp_name = (
    f"sigma_{CFG['noise_multiplier']}_"
    f"bs_{CFG['batch_size']}_"
    f"ep_{CFG['num_epochs']}_"
    f"m_{CFG['num_canaries']}_"
    f"{CFG['canary_type']}"
)

# ── Output paths ─────────────────────────────────────────────────────────────
CFG["output_dir"] = pathlib.Path(f"./experiments/{exp_name}")

CFG["output_dir"].mkdir(parents=True, exist_ok=True)

CFG["ckpt_dir"] = CFG["output_dir"] / "shadow"
CFG["ckpt_dir"].mkdir(parents=True, exist_ok=True)

# ── Display configuration ────────────────────────────────────────────────────
print("Configuration:")
for k, v in CFG.items():
    print(f"  {k:20s}: {v}")

Configuration:
  data_tar            : data/cifar-10-python.tar.gz
  data_dir            : data
  num_canaries        : 3000
  num_shadow          : 6
  canary_type         : random_labels
  seed                : 2024
  batch_size          : 4096
  max_phys_batch      : 1024
  num_epochs          : 150
  learning_rate       : 4.0
  noise_multiplier    : 1.5
  max_grad_norm       : 1.0
  delta               : 1e-05
  num_workers         : 4
  k_plus              : 450
  k_minus             : 450
  confidence_p        : 0.05
  output_dir          : experiments/sigma_1.5_bs_4096_ep_150_m_3000_random_labels
  ckpt_dir            : experiments/sigma_1.5_bs_4096_ep_150_m_3000_random_labels/shadow


---
## § 3 — Reproducibility

In [3]:
def setup_seeds(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    try:
        torch.use_deterministic_algorithms(True)
    except RuntimeError:
        pass

def shadow_seed(global_seed: int, shadow_idx: int) -> int:
    """Deterministic per-shadow seed derived from global seed."""
    return global_seed * 10000 + shadow_idx + 1

setup_seeds(CFG["seed"])
print(f"Global seed: {CFG['seed']}")

Global seed: 2024


---
## § 4 — Load CIFAR-10 from Local tar.gz (No Download)

In [4]:
import tarfile, pickle

def load_cifar10_from_tar(tar_path: pathlib.Path, data_dir: pathlib.Path):
    """
    Extract and load CIFAR-10 directly from the downloaded tar.gz.
    Returns uint8 tensors of shape (N, 3, 32, 32) and label tensors.
    No internet required.
    """
    extract_dir = data_dir / "cifar-10-batches-py"

    # Extract only if not already done
    if not extract_dir.exists():
        print(f"Extracting {tar_path} …")
        with tarfile.open(tar_path, "r:gz") as tf:
            tf.extractall(path=data_dir)
        print("Extraction complete.")
    else:
        print(f"Already extracted: {extract_dir}")

    def _load_batch(path):
        with open(path, "rb") as f:
            d = pickle.load(f, encoding="bytes")
        xs = d[b"data"].reshape(-1, 3, 32, 32)  # uint8, (N,3,32,32)
        ys = np.array(d[b"labels"], dtype=np.int64)
        return xs, ys

    # Training set: 5 batches
    train_xs_list, train_ys_list = [], []
    for i in range(1, 6):
        xs, ys = _load_batch(extract_dir / f"data_batch_{i}")
        train_xs_list.append(xs)
        train_ys_list.append(ys)
    train_xs = np.concatenate(train_xs_list, axis=0)   # (50000,3,32,32)
    train_ys = np.concatenate(train_ys_list, axis=0)   # (50000,)

    # Test set: 1 batch
    test_xs, test_ys = _load_batch(extract_dir / "test_batch")

    # Convert to torch uint8 — keep uint8 to save RAM; normalise on-the-fly
    train_xs_t = torch.from_numpy(train_xs)            # uint8 (0–255)
    train_ys_t = torch.from_numpy(train_ys)
    test_xs_t  = torch.from_numpy(test_xs)
    test_ys_t  = torch.from_numpy(test_ys)

    print(f"Train: {train_xs_t.shape}  dtype={train_xs_t.dtype}")
    print(f"Test : {test_xs_t.shape}   dtype={test_xs_t.dtype}")
    return train_xs_t, train_ys_t, test_xs_t, test_ys_t


_train_xs, _train_ys_clean, _test_xs, _test_ys = load_cifar10_from_tar(
    CFG["data_tar"], CFG["data_dir"]
)

Already extracted: data/cifar-10-batches-py
Train: torch.Size([50000, 3, 32, 32])  dtype=torch.uint8
Test : torch.Size([10000, 3, 32, 32])   dtype=torch.uint8


---
## § 5 — Transforms (Fixed random_crop)

The previous version had a `return` statement before the crop logic, making
`random_crop` only pad (producing 40×40 inputs). Fixed here.

In [5]:
def to_float(x: torch.Tensor) -> torch.Tensor:
    """uint8 CHW [0,255] → float32 CHW [0,1]"""
    if x.dtype == torch.uint8:
        return x.to(torch.float32).div_(255.0)
    return x.to(torch.float32)


def normalise(x: torch.Tensor) -> torch.Tensor:
    """Normalise float32 CHW tensor with CIFAR-10 channel stats."""
    return TFF.normalize(x, mean=list(CIFAR10_MEAN), std=list(CIFAR10_STD))


def random_crop(x: torch.Tensor, size: int = 32, padding: int = 4) -> torch.Tensor:
    """
    Pad then randomly crop back to `size`×`size`.
    FIX: previous version returned after padding — crop logic was dead code.
    Input: float32 CHW tensor. Output: float32 CHW tensor of shape (C, size, size).
    """
    # Step 1: pad all sides
    padded = TFF.pad(x, padding)          # (C, 32+2p, 32+2p)
    # Step 2: random integer offsets in [0, 2*padding]
    i = random.randint(0, 2 * padding)    # top offset
    j = random.randint(0, 2 * padding)    # left offset
    # Step 3: crop to original size
    return padded[:, i:i + size, j:j + size]


def random_hflip(x: torch.Tensor, p: float = 0.5) -> torch.Tensor:
    if random.random() < p:
        return TFF.hflip(x)
    return x


def augment(x: torch.Tensor) -> torch.Tensor:
    """Standard CIFAR-10 augmentation: random crop + random horizontal flip."""
    x = random_crop(x, size=32, padding=4)
    x = random_hflip(x, p=0.5)
    return x


def base_transform(x: torch.Tensor) -> torch.Tensor:
    """uint8 → float32 → normalised. No augmentation (used for scoring)."""
    return normalise(to_float(x))


# Sanity check: verify crop produces correct output shape
_t = torch.zeros(3, 32, 32)
_t_f = to_float(_t.to(torch.uint8))
assert random_crop(_t_f).shape == (3, 32, 32), "random_crop shape wrong!"
print("Transform functions OK. random_crop output shape: (3, 32, 32) ✓")

Transform functions OK. random_crop output shape: (3, 32, 32) ✓


---
## § 6 — Dataset: Canary Injection + IN/OUT Splits (Algorithm 1, Steps 1–4)

In [6]:
class PrivacyAuditDataset(torch.utils.data.Dataset):
    """Minimal dataset wrapper over pre-loaded tensors with optional index masking."""

    def __init__(self, xs: torch.Tensor, ys: torch.Tensor,
                 indices=None, transform=None):
        self.xs        = xs
        self.ys        = ys
        self.transform = transform
        self.idx = (
            torch.arange(len(xs)) if indices is None
            else torch.as_tensor(indices, dtype=torch.long)
        )

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        x = self.xs[self.idx[i]]
        y = self.ys[self.idx[i]]
        if self.transform is not None:
            x = self.transform(x)
        return x, y


class DataGenerator:
    """
    Manages canary injection and per-shadow IN/OUT membership.

    Algorithm 1 Step 3:
      For i in [m]: S_i ~ Uniform({-1, +1}) independently, E[S_i] = 0.
      For i in [n] \ [m]: S_i = +1 always (non-canaries always IN).

    We run `num_shadow` independent trials of this coin-flip.
    Each canary is IN (~50%) or OUT (~50%) in each shadow model independently.
    """

    def __init__(self, train_xs, train_ys_clean, test_xs, test_ys,
                 num_shadow, num_canaries, canary_type, seed):
        self.train_xs       = train_xs        # uint8 (N, 3, 32, 32)
        self.train_ys_clean = train_ys_clean  # int64 (N,)
        self.test_xs        = test_xs
        self.test_ys        = test_ys
        self.num_shadow     = num_shadow
        self.num_canaries   = num_canaries
        self.canary_type    = canary_type
        self.N              = len(train_xs)
        self.num_classes    = 10

        rng = np.random.default_rng(seed)

        # ── Step 1: choose which training examples are canaries ──────────────
        # Pick `num_canaries` indices uniformly at random (without replacement)
        self.canary_indices = rng.choice(self.N, size=num_canaries, replace=False)
        canary_set = set(self.canary_indices.tolist())

        # ── Step 2: per-shadow IN/OUT coin flips (Algorithm 1, Step 3) ──────
        # shadow_in_mask[i, s] = True if example i is IN shadow model s
        # Shape: (N, num_shadow) bool
        self.shadow_in_mask = np.ones((self.N, num_shadow), dtype=bool)
        for c_idx in self.canary_indices:
            # Each canary independently IN or OUT in each shadow model
            self.shadow_in_mask[c_idx, :] = rng.integers(0, 2, size=num_shadow).astype(bool)
        # Non-canaries: always IN (already set to True above)

        # ── Step 3: noisy labels for canaries ────────────────────────────────
        # Guarantee new label ≠ original label by sampling from remaining 9 classes
        clean = train_ys_clean.numpy()
        offsets = rng.integers(1, self.num_classes, size=self.N)  # 1..9
        self.noisy_ys = (clean + offsets) % self.num_classes      # guaranteed ≠ clean

        # Print membership balance check
        in_counts = self.shadow_in_mask[self.canary_indices].sum(axis=0)
        print(f"Canary count      : {num_canaries}")
        print(f"Canary IN per shadow (min/mean/max): "
              f"{in_counts.min()}/{in_counts.mean():.1f}/{in_counts.max()}")
        print(f"Non-canaries (always IN): {self.N - num_canaries}")

    def get_canary_indices(self) -> np.ndarray:
        return self.canary_indices

    def _apply_canary_modifications(self):
        """Return (xs, ys) tensors with canary labels/images modified."""
        xs = self.train_xs.clone()
        ys = self.train_ys_clean.clone().numpy().copy()
        ci = self.canary_indices

        if self.canary_type == "label_noise":
            # Replace canary labels with guaranteed-different random labels
            ys[ci] = self.noisy_ys[ci]

        elif self.canary_type == "blank_images":
            # Solid-colour images: one random RGB colour per canary, random label
            rng2 = np.random.default_rng(42)
            colors = rng2.integers(0, 256, size=(len(ci), 3, 1, 1), dtype=np.uint8)
            xs[ci] = torch.from_numpy(
                np.broadcast_to(colors, (len(ci), 3, 32, 32)).copy()
            )
            ys[ci] = rng2.integers(0, self.num_classes, size=len(ci))

        # "clean" canary type: no modification (weakest signal, baseline)
        return xs, torch.from_numpy(ys)

    def build_train_data(self, shadow_idx: int,
                         transform=None) -> PrivacyAuditDataset:
        """Return the IN-subset for shadow model `shadow_idx`."""
        xs, ys = self._apply_canary_modifications()
        in_idx = np.where(self.shadow_in_mask[:, shadow_idx])[0]
        return PrivacyAuditDataset(xs, ys, indices=in_idx, transform=transform)

    def build_full_data(self, transform=None) -> PrivacyAuditDataset:
        """All 50k examples with canary modifications (for scoring)."""
        xs, ys = self._apply_canary_modifications()
        return PrivacyAuditDataset(xs, ys, transform=transform)

    def is_in_mask(self, shadow_idx: int) -> torch.Tensor:
        """Bool tensor (N,) — True if example is IN for this shadow model."""
        return torch.from_numpy(self.shadow_in_mask[:, shadow_idx])


# ── Instantiate ───────────────────────────────────────────────────────────────
data_gen = DataGenerator(
    train_xs=_train_xs, train_ys_clean=_train_ys_clean,
    test_xs=_test_xs,   test_ys=_test_ys,
    num_shadow=CFG["num_shadow"],
    num_canaries=CFG["num_canaries"],
    canary_type=CFG["canary_type"],
    seed=CFG["seed"],
)

Canary count      : 3000
Canary IN per shadow (min/mean/max): 1490/1507.3/1522
Non-canaries (always IN): 47000


---
## § 7 — WideResNet-16-4 with GroupNorm (Opacus-compatible)

In [7]:
class WideBlock(nn.Module):
    """Pre-activation wide residual block."""

    def __init__(self, fin, fout, stride, expand, norm_fn):
        super().__init__()
        self.n1 = norm_fn(fin)
        self.c1 = nn.Conv2d(fin, fout, 3, 1, 1, bias=False)
        self.n2 = norm_fn(fout)
        self.c2 = nn.Conv2d(fout, fout, 3, stride, 1, bias=False)
        if stride != 1 or expand:
            self.skip = nn.Conv2d(fin, fout, 1, stride, bias=False)
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        h = self.c1(F.relu(self.n1(x)))
        h = self.c2(F.relu(self.n2(h)))
        return h + self.skip(x)


class WideResNet(nn.Module):
    """
    WideResNet-depth-widen with GroupNorm.
    depth=16, widen=4 → 2.75M params, matches paper §6 experiment setup.
    GroupNorm replaces BatchNorm for Opacus compatibility.
    """

    def __init__(self, depth=16, widen=4, num_classes=10):
        super().__init__()
        assert (depth - 4) % 6 == 0, "depth must satisfy (depth-4) % 6 == 0"
        nb = (depth - 4) // 6
        fs = (16, 16 * widen, 32 * widen, 64 * widen)

        def gn(c): return nn.GroupNorm(min(16, c), c)

        self.conv0  = nn.Conv2d(3, fs[0], 3, 1, 1, bias=False)
        self.group1 = self._make_layer(fs[0], fs[1], 1,  nb, gn)
        self.group2 = self._make_layer(fs[1], fs[2], 2,  nb, gn)
        self.group3 = self._make_layer(fs[2], fs[3], 2,  nb, gn)
        self.norm_out = gn(fs[3])
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.flat   = nn.Flatten()
        self.fc     = nn.Linear(fs[3], num_classes)
        self._init_weights()

    def _make_layer(self, fin, fout, stride, n_blocks, norm_fn):
        layers = []
        for i in range(n_blocks):
            layers.append(WideBlock(
                fin if i == 0 else fout,
                fout,
                stride if i == 0 else 1,
                fin != fout,
                norm_fn,
            ))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                fan_in = max(1, m.weight.size(1) * m.weight.size(2) * m.weight.size(3))
                nn.init.trunc_normal_(m.weight, std=1.0 / fan_in ** 0.5)
            elif isinstance(m, nn.GroupNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=1.0 / m.weight.size(1) ** 0.5)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.conv0(x)
        h = self.group1(h)
        h = self.group2(h)
        h = self.group3(h)
        h = F.relu(self.norm_out(h))
        return self.fc(self.flat(self.pool(h)))


def build_model() -> nn.Module:
    return WideResNet(depth=16, widen=4, num_classes=10).to(DEVICE)


# Smoke test
_m = build_model()
_x = torch.randn(2, 3, 32, 32, device=DEVICE)
assert _m(_x).shape == (2, 10)
n_params = sum(p.numel() for p in _m.parameters())
print(f"WideResNet-16-4: {n_params:,} parameters ✓")
del _m, _x

WideResNet-16-4: 2,834,906 parameters ✓


---
## § 8 — Training Datasets & EMA Helper

In [8]:
class TrainDataset(torch.utils.data.Dataset):
    """Wraps PrivacyAuditDataset; converts uint8→float, normalises, augments."""

    def __init__(self, ds: PrivacyAuditDataset):
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        x, y = self.ds[i]
        x = to_float(x)
        x = augment(x)      # crop + hflip (32×32 output verified)
        x = normalise(x)
        return x, y


class ScoreDataset(torch.utils.data.Dataset):
    """
    Features-only dataset used during scoring.
    NO augmentation — deterministic transform for reproducible loss values.
    """

    def __init__(self, ds: PrivacyAuditDataset):
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        x, _ = self.ds[i]
        return base_transform(x)   # uint8 → float → normalised, no augmentation


# ── EMA ───────────────────────────────────────────────────────────────────────
def create_ema(model: nn.Module) -> nn.Module:
    ema = copy.deepcopy(model)
    for p in ema.parameters():
        p.detach_()
    return ema


def update_ema(model: nn.Module, ema: nn.Module, step: int, decay=0.9999):
    eff = min(decay, (1 + step) / (10 + step))
    with torch.no_grad():
        for mp, ep in zip(model.parameters(), ema.parameters()):
            ep.sub_((1.0 - eff) * (ep - mp))
        for mb, eb in zip(model.buffers(), ema.buffers()):
            eb.copy_(mb)


print("TrainDataset / ScoreDataset / EMA defined.")

TrainDataset / ScoreDataset / EMA defined.


---
## § 9 — Train One Shadow Model (Algorithm 1, Steps 3–5 + Algorithm 3)

**Critical:** `w_0` (the initial weights before training) is saved and used
to compute the loss-difference score in § 10.

In [9]:
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from opacus.utils.batch_memory_manager import BatchMemoryManager


def train_shadow_model(shadow_idx: int, cfg: dict,
                       data_gen: DataGenerator,
                       verbose: bool = True):
    """
    Train one shadow model with DP-SGD.

    Returns
    -------
    ema_model : nn.Module
        EMA of the trained model (w_T). Used for final-loss scoring.
    w0_state  : dict
        state_dict of the model BEFORE any training (w_0).
        Required for Score(x) = loss(w_0, x) − loss(w_T, x).

    Checkpointing: if the checkpoint already exists, load and return immediately.
    """
    ckpt_dir  = cfg["ckpt_dir"] / str(shadow_idx)
    ckpt_ema  = ckpt_dir / "ema_model.pt"
    ckpt_w0   = ckpt_dir / "w0_state.pt"

    if ckpt_ema.exists() and ckpt_w0.exists():
        print(f"  [shadow {shadow_idx}] Checkpoint found — loading.")
        ema   = torch.load(ckpt_ema, map_location=DEVICE, weights_only=False)
        w0_sd = torch.load(ckpt_w0,  map_location="cpu",  weights_only=True)
        return ema, w0_sd

    ckpt_dir.mkdir(parents=True, exist_ok=True)
    setup_seeds(shadow_seed(cfg["seed"], shadow_idx))

    # ── Dataset ──────────────────────────────────────────────────────────────
    raw_ds  = data_gen.build_train_data(shadow_idx)
    train_ds = TrainDataset(raw_ds)
    loader   = torch.utils.data.DataLoader(
        train_ds,
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=cfg["num_workers"],
        drop_last=False,
        persistent_workers=(cfg["num_workers"] > 0),
        pin_memory=True,
    )

    # ── Model ────────────────────────────────────────────────────────────────
    model = build_model()
    model = ModuleValidator.fix(model)
    errs  = ModuleValidator.validate(model, strict=False)
    if errs:
        raise RuntimeError(f"Opacus validation errors: {errs}")

    # ── Save w_0: initial weights BEFORE any gradient update ─────────────────
    # This is the reference point for the loss-difference score (Algorithm 3 §8)
    w0_state = copy.deepcopy(model.state_dict())
    # Move all tensors to CPU for storage efficiency
    w0_state_cpu = {k: v.cpu() for k, v in w0_state.items()}
    torch.save(w0_state_cpu, ckpt_w0)

    # ── Optimiser + Opacus ───────────────────────────────────────────────────
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=cfg["learning_rate"],
        momentum=0.0,
        weight_decay=0.0,
    )
    loss_fn = nn.CrossEntropyLoss(reduction="mean")

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        pe = PrivacyEngine()
        model, optimizer, loader = pe.make_private(
            module=model,
            optimizer=optimizer,
            data_loader=loader,
            noise_multiplier=cfg["noise_multiplier"],
            max_grad_norm=cfg["max_grad_norm"],
        )

    ema        = create_ema(model)
    num_updates = 0
    log_every  = max(1, cfg["num_epochs"] // 10)

    # ── Training loop ─────────────────────────────────────────────────────────
    for epoch in range(cfg["num_epochs"]):
        model.train()
        with BatchMemoryManager(
            data_loader=loader,
            max_physical_batch_size=cfg["max_phys_batch"],
            optimizer=optimizer,
        ) as opt_loader:
            epoch_loss = epoch_n = 0

            for bx, by in opt_loader:
                bx = bx.to(DEVICE)
                by = by.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)

                loss = loss_fn(model(bx), by)
                loss.backward()

                will_update = not optimizer._check_skip_next_step(pop_next=False)
                optimizer.step()
                if will_update:
                    num_updates += 1
                    update_ema(model, ema, num_updates)

                epoch_loss += loss.item() * bx.size(0)
                epoch_n    += bx.size(0)

        if verbose or (epoch + 1) % log_every == 0:
            eps = pe.get_epsilon(cfg["delta"])
            print(f"    epoch {epoch+1:3d}/{cfg['num_epochs']}  "
                  f"loss={epoch_loss/epoch_n:.4f}  ε={eps:.4f}")

    ema.eval()
    torch.save(ema, ckpt_ema)
    print(f"  [shadow {shadow_idx}] saved → {ckpt_dir}")
    return ema, w0_state_cpu


print("train_shadow_model() defined.")

train_shadow_model() defined.


---
## § 10 — Train Model

In [10]:
import os
import time

# Storage: maps shadow_idx → (ema_model, w0_state_dict)
trained_models = {}   # shadow_idx → ema nn.Module
w0_states      = {}   # shadow_idx → state_dict (CPU)

print("Starting/Resuming Training Loop on GPU 1...")

# 🟢 FIX: Only train the last 7 models on GPU 1 (Indices 8 through 14)
for s in range(4,6): 
    print(f"\n─── Shadow model {s+1}/{CFG['num_shadow']} ───")
    t0 = time.time()
    
    ema, w0_sd = train_shadow_model(s, CFG, data_gen, verbose=False)
    
    trained_models[s] = ema
    w0_states[s]      = w0_sd
    
    # 🟢 PARANOID-PROOF LOGGING: Saves to a GPU-1 specific file
    gpu_id = os.environ.get("CUDA_VISIBLE_DEVICES", "1")
    log_file = CFG["output_dir"] / f"completed_models_GPU{gpu_id}.txt"
    with open(log_file, "a") as f:
        f.write(f"Model {s} completed at {time.ctime()}\n")
        
    print(f"  Done in {(time.time()-t0)/60:.1f} min")

print("\nAll assigned models for GPU 1 (Models 8-14) are complete.")

Starting/Resuming Training Loop on GPU 1...

─── Shadow model 5/6 ───
    epoch  15/150  loss=2.0482  ε=3.9155
    epoch  30/150  loss=1.8333  ε=5.6579
    epoch  45/150  loss=1.7468  ε=7.0824
    epoch  60/150  loss=1.6824  ε=8.3404
    epoch  75/150  loss=1.8023  ε=9.4912
    epoch  90/150  loss=1.6090  ε=10.5655
    epoch 105/150  loss=1.5933  ε=11.5816
    epoch 120/150  loss=1.6171  ε=12.5515
    epoch 135/150  loss=1.4565  ε=13.4835
    epoch 150/150  loss=1.4377  ε=14.3838
  [shadow 4] saved → experiments/sigma_1.5_bs_4096_ep_150_m_3000_random_labels/shadow/4
  Done in 53.4 min

─── Shadow model 6/6 ───
    epoch  15/150  loss=1.9673  ε=3.9155
    epoch  30/150  loss=1.9067  ε=5.6579
    epoch  45/150  loss=1.7603  ε=7.0824
    epoch  60/150  loss=1.6681  ε=8.3404
    epoch  75/150  loss=1.6233  ε=9.4912
    epoch  90/150  loss=1.6000  ε=10.5655
    epoch 105/150  loss=1.4822  ε=11.5816
    epoch 120/150  loss=1.5064  ε=12.5515
    epoch 135/150  loss=1.4655  ε=13.4835
    epoch

---
## § 11 — Score Computation (Algorithm 3, Line 9)

**Black-box score (paper §6.2, input-space attack):**
```
Score(xᵢ, w) = loss(w_0, xᵢ) − loss(w_T, xᵢ)
```
- `w_0` = initial weights saved before training (§ 9)
- `w_T` = EMA of final trained weights
- Higher score → greater loss reduction → model memorised xᵢ → xᵢ likely IN

The key insight: initial loss cancels out the per-example 'difficulty' bias,
so the score measures *relative* memorisation rather than absolute loss level.

In [11]:
@torch.no_grad()
def compute_losses(model: nn.Module, dataset, labels: torch.Tensor,
                   batch_size: int = 512) -> torch.Tensor:
    """
    Run model on `dataset` (features only) and return per-example cross-entropy.
    Shape: (N,) float32 on CPU.
    """
    model.eval()
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=CFG["num_workers"],
        persistent_workers=(CFG["num_workers"] > 0),
        pin_memory=True,
    )
    losses = []
    start  = 0
    for bx in loader:
        end = start + bx.size(0)
        by  = labels[start:end].to(DEVICE)
        logits = model(bx.to(DEVICE))
        loss = F.cross_entropy(logits, by, reduction="none").cpu()
        losses.append(loss)
        start = end
    return torch.cat(losses, dim=0)   # (N,)


print("compute_losses() defined.")

compute_losses() defined.


In [12]:
# ── Build the full dataset (all 50k examples, with canary modifications) ──────
print("Building full dataset (canary modifications applied) …")
full_ds  = data_gen.build_full_data()          # PrivacyAuditDataset (50k)
score_ds = ScoreDataset(full_ds)               # features-only, base_transform

# Collect labels for cross-entropy computation
all_ys = torch.stack([full_ds[i][1] for i in range(len(full_ds))])  # (50000,)

canary_idx   = torch.from_numpy(data_gen.get_canary_indices().astype("int64"))
N_CANARY     = len(canary_idx)
N_SHADOW     = CFG["num_shadow"]

print(f"Scoring {N_CANARY} canaries across {N_SHADOW} shadow models …\n")

# ── Storage ───────────────────────────────────────────────────────────────────
# shadow_scores[s, i] = Score(x_i, shadow s) = loss(w_0,x_i) - loss(w_T,x_i)
shadow_ids = sorted(trained_models.keys())

shadow_scores = {}
membership    = {}

# ── Score loop ────────────────────────────────────────────────────────────────
for s in sorted(trained_models.keys()):
    print(f"  Shadow {s:2d}: ", end="", flush=True)

    ema_model = trained_models[s]
    w0_sd     = w0_states[s]

    # ── Build a temporary model loaded with w_0 weights ───────────────────
    # We need loss(w_0, x) for every canary.
    # w_0 was saved before Opacus wrapping, so load into a fresh base model.
    w0_model = build_model()
    w0_model = ModuleValidator.fix(w0_model)
    # Opacus wraps the model in GradSampleModule; w0 was saved from the raw model
    # so we load into the raw model directly.
    w0_model.load_state_dict(w0_sd, strict=True)
    w0_model.eval()

    # ── Compute losses ────────────────────────────────────────────────────
    # loss(w_0, x) for all 50k examples
    initial_loss = compute_losses(w0_model, score_ds, all_ys)   # (50000,)
    del w0_model
    torch.cuda.empty_cache()

    # loss(w_T, x) for all 50k examples (w_T = EMA final model)
    final_loss = compute_losses(ema_model, score_ds, all_ys)     # (50000,)

    # ── Score = loss(w_0, x) − loss(w_T, x) (Algorithm 3, line 9) ────────
    # Positive score → loss decreased → model memorised x → x likely IN
    score_full = initial_loss - final_loss                         # (50000,)

    # ── Extract canary scores and membership ──────────────────────────────
    shadow_scores[s] = score_full[canary_idx].cpu()
    membership[s]    = data_gen.is_in_mask(s)[canary_idx].cpu()

    # Diagnostics
    in_mask  = membership[s]
    nin      = in_mask.sum().item()
    s_in     = shadow_scores[s][in_mask].mean().item()
    s_out    = shadow_scores[s][~in_mask].mean().item()
    print(f"IN={nin:4d}  OUT={N_CANARY-nin:4d}  "
          f"score_mean IN={s_in:+.4f}  OUT={s_out:+.4f}  "
          f"Δ={s_in-s_out:+.4f}")

    # ── Incremental save ──────────────────────────────────────────────────
    torch.save(shadow_scores, CFG["ckpt_dir"] / "shadow_scores.pt")
    torch.save(membership,    CFG["ckpt_dir"] / "membership.pt")
    np.save(CFG["ckpt_dir"] / "canary_idx.npy",    canary_idx.numpy())

print("\nScoring complete.")

Building full dataset (canary modifications applied) …
Scoring 3000 canaries across 6 shadow models …

  Shadow  4: IN=1490  OUT=1510  score_mean IN=+1.2497  OUT=+1.1661  Δ=+0.0836
  Shadow  5: IN=1517  OUT=1483  score_mean IN=+1.2017  OUT=+1.2440  Δ=-0.0423

Scoring complete.


---
## § 12 — Statistical Audit (Corollary 5.4)

**Algorithm 1, Steps 7–8:**
Sort scores → guess T_i = +1 for top-k (likely IN), T_i = -1 for bottom-k (likely OUT).

**Corollary 5.4:** Given v correct guesses out of r total guesses, find the largest ε
such that the p-value (probability of observing ≥v correct guesses under the
null hypothesis of (ε, δ)-DP) is < p. That ε is our lower bound.

In [13]:
def p_value_DP_audit(m: int, r: int, v: int, eps: float, delta: float) -> float:
    """
    P-value of observing ≥v correct guesses (out of r) under the null
    hypothesis that the algorithm is (eps, delta)-DP.

    From Corollary 5.4 of Steinke et al. 2023.
    The dominating distribution W* is Binomial(r, e^eps / (e^eps + 1)).

    Parameters
    ----------
    m     : total number of canaries (for the delta correction term)
    r     : total guesses made (k_plus + k_minus)
    v     : number of correct guesses
    eps   : candidate privacy parameter
    delta : privacy parameter δ
    """
    assert 0 <= v <= r <= m
    assert eps >= 0
    assert 0 <= delta <= 1

    q = math.exp(eps) / (math.exp(eps) + 1.0)   # = 1 / (1 + e^{-eps})

    # β = P[Binomial(r, q) ≥ v]  (survival function)
    beta = scipy.stats.binom.sf(v - 1, r, q)

    # α correction for approximate DP (δ > 0 term from Theorem 5.2 eq. 9)
    alpha    = 0.0
    running  = 0.0
    for i in range(1, v + 1):
        running += scipy.stats.binom.pmf(v - i, r, q)
        if running > i * alpha:
            alpha = running / i

    p = beta + alpha * delta * 2 * m
    return min(p, 1.0)


def get_eps_lower_bound(m: int, r: int, v: int, delta: float,
                        p: float = 0.05) -> float:
    """
    Binary search for the largest ε such that p_value_DP_audit(..., eps) < p.
    This ε is the empirical lower bound with confidence 1-p.
    """
    assert 0 <= v <= r <= m
    assert 0 < p < 1

    # Quick exit: if even ε=0 gives p ≥ threshold, return 0
    if p_value_DP_audit(m, r, v, 0.0, delta) >= p:
        return 0.0

    eps_lo, eps_hi = 0.0, 1.0
    # Expand upper bracket until p-value crosses threshold
    while p_value_DP_audit(m, r, v, eps_hi, delta) < p:
        eps_hi += 1.0

    # 50 iterations → precision < 1e-14
    for _ in range(50):
        eps_mid = (eps_lo + eps_hi) / 2.0
        if p_value_DP_audit(m, r, v, eps_mid, delta) < p:
            eps_lo = eps_mid
        else:
            eps_hi = eps_mid

    return eps_lo


print("Audit functions defined.")

Audit functions defined.


In [14]:
# ── Audit parameters ──────────────────────────────────────────────────────────
M  = CFG["num_canaries"]
kp = CFG["k_plus"]
km = CFG["k_minus"]
r  = kp + km
p  = CFG["confidence_p"]
d  = CFG["delta"]

print(f"Audit: m={M}  k+={kp}  k-={km}  r={r}  p={p}  δ={d}")
print(f"Chance baseline: {r/2}/{r} = 0.5000\n")

per_shadow_eps = []
per_shadow_acc = []

for s in range(N_SHADOW):
    scores_s   = shadow_scores[s]            # (m,)
    member_s   = membership[s]               # (m,) bool

    # Algorithm 1 Step 7: sort scores ascending → low = likely OUT, high = likely IN
    sorted_idx = torch.argsort(scores_s)

    # Step 8: T_i = -1 for bottom k_minus (guessed OUT), +1 for top k_plus (guessed IN)
    bottom_k = sorted_idx[:km]    # lowest scores  → guess OUT
    top_k    = sorted_idx[-kp:]   # highest scores → guess IN

    correct_out = (~member_s[bottom_k]).sum().item()   # guessed OUT, actually OUT
    correct_in  = member_s[top_k].sum().item()          # guessed IN,  actually IN
    v           = correct_in + correct_out

    acc    = v / r
    eps_lb = get_eps_lower_bound(M, r, v, d, p)

    per_shadow_acc.append(acc)
    per_shadow_eps.append(eps_lb)

    print(f"  shadow {s:2d} | v={v:3d}/{r}  acc={acc:.4f}  "
          f"correct_in={correct_in}/{kp}  correct_out={correct_out}/{km}  "
          f"ε≥{eps_lb:.4f}")

best_eps = max(per_shadow_eps)
avg_eps  = float(np.mean(per_shadow_eps))
print(f"\nBest ε lower bound : ε ≥ {best_eps:.4f}")
print(f"Avg  ε lower bound : ε ≥ {avg_eps:.4f}")

Audit: m=3000  k+=450  k-=450  r=900  p=0.05  δ=1e-05
Chance baseline: 450.0/900 = 0.5000



KeyError: 0

---
## § 13 — Sweep k to Find Best Audit Configuration

The paper sweeps k_plus = k_minus = k over a range and reports the best ε.
This trades off: larger k → more guesses → more statistical power but lower
accuracy per guess.  The optimal k depends on the score separation.

In [ ]:
k_vals = list(range(10, min(M // 2 + 1, 501), 10))
sweep_table = []   # (k, best_eps_across_shadows)

for k in k_vals:
    best_k_eps = 0.0
    for s in range(N_SHADOW):
        sorted_idx = torch.argsort(shadow_scores[s])
        bot = sorted_idx[:k]
        top = sorted_idx[-k:]
        v_k = (membership[s][top].sum() + (~membership[s][bot]).sum()).item()
        r_k = 2 * k
        if 0 <= v_k <= r_k <= M:
            best_k_eps = max(best_k_eps, get_eps_lower_bound(M, r_k, v_k, d, p))
    sweep_table.append((k, best_k_eps))

sweep_k   = [t[0] for t in sweep_table]
sweep_eps = [t[1] for t in sweep_table]
best_k    = sweep_k[int(np.argmax(sweep_eps))]
best_eps_sweep = max(sweep_eps)

print(f"Sweep result: best k={best_k}  →  ε ≥ {best_eps_sweep:.4f}")

---
## § 14 — Theoretical Upper Bound (Opacus accountant)

In [ ]:
# Re-run Opacus accountant with the same hyperparameters to get theoretical ε.
# We step the accountant manually — no actual training needed.

def compute_theoretical_eps(cfg: dict, n_train: int) -> float:
    """Compute the theoretical ε upper bound using Opacus's RDP accountant."""
    from opacus import PrivacyEngine

    pe = PrivacyEngine()
    _m   = build_model()
    _m   = ModuleValidator.fix(_m)
    _opt = torch.optim.SGD(_m.parameters(), lr=0.01)

    # Build a dummy dataloader of the right size
    dummy_ds = torch.utils.data.TensorDataset(
        torch.zeros(n_train, 3, 32, 32),
        torch.zeros(n_train, dtype=torch.long)
    )
    dummy_dl = torch.utils.data.DataLoader(
        dummy_ds, batch_size=cfg["batch_size"], shuffle=False
    )

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        _, _, priv_dl = pe.make_private(
            module=_m, optimizer=_opt, data_loader=dummy_dl,
            noise_multiplier=cfg["noise_multiplier"],
            max_grad_norm=cfg["max_grad_norm"],
        )

    steps_per_epoch = len(priv_dl)
    sample_rate     = cfg["batch_size"] / n_train
    for _ in range(cfg["num_epochs"] * steps_per_epoch):
        pe.accountant.step(
            noise_multiplier=cfg["noise_multiplier"],
            sample_rate=sample_rate,
        )
    del _m, _opt, dummy_dl, dummy_ds
    return pe.get_epsilon(cfg["delta"])


# Average IN-set size (non-canaries always IN, canaries ~50%)
avg_n_train = int(data_gen.shadow_in_mask.sum(0).mean())
theoretical_eps = compute_theoretical_eps(CFG, avg_n_train)
print(f"Theoretical ε upper bound: ε ≤ {theoretical_eps:.4f}  (δ={CFG['delta']})")

---
## § 15 — Results Summary

In [ ]:
# ── Score separation statistics ───────────────────────────────────────────────
all_in_scores  = shadow_scores[membership].numpy()
all_out_scores = shadow_scores[~membership].numpy()

DIV = "═" * 64
print(DIV)
print("   PRIVACY AUDIT RESULTS  (Steinke, Nasr & Jagielski 2023)")
print(DIV)
print(f"\n── Configuration ──────────────────────────────────────────")
print(f"  Canaries (m)          : {CFG['num_canaries']}")
print(f"  Canary type           : {CFG['canary_type']}")
print(f"  Shadow models         : {CFG['num_shadow']}")
print(f"  Epochs / Batch size   : {CFG['num_epochs']} / {CFG['batch_size']}")
print(f"  Noise multiplier σ    : {CFG['noise_multiplier']}")
print(f"  Max grad norm c       : {CFG['max_grad_norm']}")
print(f"  δ                     : {CFG['delta']}")
print(f"  Score formula         : loss(w_0, x) − loss(w_T, x)  [Alg. 3 §8]")

print(f"\n── Score Separation ────────────────────────────────────────")
print(f"  IN  mean / std        : {all_in_scores.mean():+.4f} / {all_in_scores.std():.4f}")
print(f"  OUT mean / std        : {all_out_scores.mean():+.4f} / {all_out_scores.std():.4f}")
print(f"  Δ mean (IN − OUT)     : {all_in_scores.mean()-all_out_scores.mean():+.4f}")

print(f"\n── Audit Results ───────────────────────────────────────────")
print(f"  k+ / k-               : {kp} / {km}")
print(f"  Best audit accuracy   : {max(per_shadow_acc):.4f}  (chance = 0.5000)")
print(f"  Avg  audit accuracy   : {np.mean(per_shadow_acc):.4f}")

print(f"\n── Privacy Bounds ──────────────────────────────────────────")
print(f"  ε upper bound (Opacus): ε ≤ {theoretical_eps:.4f}")
print(f"  ε lower bound (best)  : ε ≥ {best_eps:.4f}  (95% confidence)")
print(f"  ε lower bound (sweep) : ε ≥ {best_eps_sweep:.4f}  (best k={best_k})")
print(f"  ε lower bound (avg)   : ε ≥ {avg_eps:.4f}")
print(f"  Privacy gap           : [{best_eps:.4f}, {theoretical_eps:.4f}]")
print(f"\n{DIV}")

# Save JSON
result = dict(
    theoretical_eps=theoretical_eps,
    best_eps=best_eps,
    avg_eps=avg_eps,
    best_eps_sweep=best_eps_sweep,
    best_k=int(best_k),
    per_shadow_eps=per_shadow_eps,
    per_shadow_acc=per_shadow_acc,
    score_delta_mean=float(all_in_scores.mean() - all_out_scores.mean()),
    cfg={k: str(v) for k, v in CFG.items()},
)
out_path = CFG["output_dir"] / "audit_results.json"
with open(out_path, "w") as f:
    json.dump(result, f, indent=2)
print(f"Results saved → {out_path}")

---
## § 16 — Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── (a) Score distribution IN vs OUT ─────────────────────────────────────────
ax = axes[0]
ax.hist(all_in_scores,  bins=50, alpha=0.65, color="steelblue",
        label=f"IN  canaries (n={len(all_in_scores):,})", density=True)
ax.hist(all_out_scores, bins=50, alpha=0.65, color="tomato",
        label=f"OUT canaries (n={len(all_out_scores):,})", density=True)
ax.axvline(0, color="black", lw=1.0, ls="--", label="Score = 0")
ax.set_title("Score Distribution: IN vs OUT", fontsize=12)
ax.set_xlabel(r"Score  =  $\ell(w_0, x) - \ell(w_T, x)$", fontsize=10)
ax.set_ylabel("Density")
ax.text(0.97, 0.97,
        f"Δ Mean = {all_in_scores.mean()-all_out_scores.mean():.4f}",
        ha="right", va="top", transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle="round", fc="white", alpha=0.8))
ax.legend(fontsize=9)

# ── (b) ε lower bound per shadow model ───────────────────────────────────────
ax = axes[1]
colors = ["steelblue" if e == best_eps else "#aec6e8" for e in per_shadow_eps]
ax.bar(range(N_SHADOW), per_shadow_eps, color=colors)
ax.axhline(avg_eps, color="darkorange", lw=1.5, ls="--",
           label=f"Mean ε = {avg_eps:.3f}")
ax.axhline(best_eps, color="red", lw=1.5, ls=":",
           label=f"Best ε = {best_eps:.3f}")
ax.axhline(theoretical_eps, color="black", lw=1.2, ls="-",
           label=f"Theory ε ≤ {theoretical_eps:.3f}")
ax.set_title("ε Lower Bound per Shadow Model", fontsize=12)
ax.set_xlabel("Shadow model index")
ax.set_ylabel("Empirical ε lower bound")
ax.legend(fontsize=8)

# ── (c) Sweep: ε lower bound vs k ────────────────────────────────────────────
ax = axes[2]
ax.plot(sweep_k, sweep_eps, marker="o", ms=3, color="steelblue", lw=1.5)
ax.axvline(best_k, color="red", ls=":", lw=1.5, label=f"Best k = {best_k}")
ax.axhline(theoretical_eps, color="black", lw=1.2, ls="-",
           label=f"Theory ε ≤ {theoretical_eps:.3f}")
ax.set_title("ε Lower Bound vs Guess Budget k", fontsize=12)
ax.set_xlabel("k  (k+ = k- = k)")
ax.set_ylabel("ε lower bound")
ax.legend(fontsize=9)

plt.tight_layout()
fig_path = CFG["output_dir"] / "audit_plots.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved → {fig_path}")

---
## § 17 — Notes on Expected Results & Scaling

### What to expect with this configuration

| Config | Theoretical ε | Expected ε_LB (black-box) |
|---|---|---|
| σ=3.0, batch=4096, epochs=150 | ≈ 5.3 | 0.2–0.8 (black-box is weak) |
| σ=1.0, batch=4096, epochs=150 | ≈ 14  | 0.5–2.0 |
| σ=0.4, batch=4096, epochs=60  | ≈ 100 | 1–4 |

Black-box input-space auditing always gives loose bounds — this is expected.
The paper's strong results (ε ≥ 1.8 at theory ε = 4) use white-box gradient attacks.
Increasing `num_canaries` to 5000 and `num_shadow` to 32 will tighten the bound.

### If ε_LB = 0.0 (no signal detected)
1. Check the score histogram — does IN mean > OUT mean? (Even 0.01 difference matters)
2. Try `canary_type = "blank_images"` — strongest signal, model can memorise solid colours easily
3. Reduce `noise_multiplier` (e.g. 1.5) to let the model learn more under DP
4. Increase `num_canaries` (e.g. 2000) — more statistical power per shadow model
5. Increase `num_shadow` — more independent trials to find the best ε_LB

### Key formula reference (Algorithm 3, line 9)
```
Score(xᵢ, w₀, w_T) = loss(w₀, xᵢ) − loss(w_T, xᵢ)
```
- `w₀` = saved before first gradient step (this notebook § 9)
- `w_T` = EMA of weights after all epochs
- For IN examples (were trained on): loss decreases → score is positive
- For OUT examples (never seen): loss barely changes → score near zero
- The difference: IN scores should be systematically higher than OUT scores